# Day 6: Session 6A - The Join Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/6a_joining_data.html)

Date: 09/08/2026

In [2]:
import pandas as pd

url = 'https://eds-217-essential-python.github.io/data/openaq_goleta_measurments.csv'
goleta = pd.read_csv(url)

goleta['parameter'].value_counts()

parameter
pm25    734
o3      711
pm10    517
Name: count, dtype: int64

In [ ]:
# creating two tables from the main df
o3 = goleta[goleta['parameter'] == 'o3'].copy()
pm25 = goleta[goleta['parameter'] == 'pm25'].copy()

print(o3.shape)
print(pm25.shape)

# now trim the dfs to the columns of interest
# and also you have to rename columns that have the same name
# across dfs because if they share the same name, the resulting
# df from the merge will be messed up a bit
o3 = o3[['datetimeLocal', 'value']]
o3 = o3.rename(columns={'value': 'o3_ppm'})

pm25 = pm25[['datetimeLocal', 'value']]
pm25 = pm25.rename(columns={'value': 'pm25_ugm3'})

o3.head()

(711, 15)
(734, 15)


,datetimeLocal,o3_ppm
0,2024-07-11T18:00:00-07:00,0.025
1,2024-07-11T19:00:00-07:00,0.028
2,2024-07-11T20:00:00-07:00,0.029
3,2024-07-11T21:00:00-07:00,0.027
4,2024-07-11T22:00:00-07:00,0.026


In [ ]:
# now we can merge the two dfs using pd.merge()

paired = pd.merge(o3, pm25, on = 'datetimeLocal')
paired.head()
paired.shape
# so what we just did was an inner join of the two data sets
# which creates a df with rows where each initial df 
# (o3 and pm25) share the same datetimeLocal value
# the default of pd.merge() is inner merge

(704, 3)

In [9]:
# now we can do try a left merge
left = pd.merge(o3, pm25, on='datetimeLocal', how='left')

left.shape
# setting how= to 'left' creates the left merge
# left is the first df listed (o3 in this case) 
# and riht is the second one listed (pm25)

# we should check if there are NAs, which there will be when
# left or right merging sometiems
left.isnull().sum()

datetimeLocal    0
o3_ppm           0
pm25_ugm3        7
dtype: int64

In [10]:
# now a right one
right = pd.merge(o3, pm25, on='datetimeLocal', how='right')

print(right.shape)
print(right.isnull().sum())

(734, 3)
datetimeLocal     0
o3_ppm           30
pm25_ugm3         0
dtype: int64


In [11]:
# and now an outer merge, which just merges all data
outer = pd.merge(o3, pm25, on='datetimeLocal', how='outer')

print(outer.shape)
print(outer.isnull().sum())

(741, 3)
datetimeLocal     0
o3_ppm           30
pm25_ugm3         7
dtype: int64


In [ ]:
pm10 = goleta[goleta['parameter'] == 'pm10']
pm10 = pm10[['datetimeLocal','value']]
pm10 = pm10.rename(columns= {'value':'pm10v'})

pmsleft = pd.merge(pm25, pm10, on = 'datetimeLocal', how = 'left')
pmsleft
pmsinner = pd.merge(pm25, pm10, on = 'datetimeLocal')
pmsinner
pmsleft.shape[0]-pmsinner.shape[0] # this is how many pm25 readings DONT have an accompanying pm10 reading


222

In [ ]:
missing_o3 = right[right['o3_ppm'].isnull()].copy()

missing_o3['datetimeLocal'].str[11:13].value_counts()
# the .str[11:13] refers to the 11-13 character in the date format found
# in the datetimeLocal column, and the 11-13th characters are the
# time of day
# when looking at which rows have no data for o3, you
# see that they all occur at 03:00, kinda weird!

# its good to look and check which values didnt make it through the merge,
# because maybe theres a specific reason

datetimeLocal
03    30
Name: count, dtype: int64

In [41]:
# doing the same thing for the left join
missingpm25 = left[left['pm25_ugm3'].isnull()].copy()
missingpm25['datetimeLocal'].str[11:13].value_counts()


datetimeLocal
14    2
15    2
04    1
05    1
06    1
Name: count, dtype: int64